# Statistical Evaluation of VUCCA Cache Results

This notebook demonstrates the evaluation pipeline for the Visual Uncertainty-Aware Conformal Cache Admission (VUCCA) method. It loads experiment results, computes aggregate metrics (hit ratio, throughput, average latency, coverage error), performs paired statistical tests against LRU, TinyLFU, and ARC2 baselines, and reports data-quality flags.

**What this artifact does**
- Parses cached experiment outputs organized by workload regime and admission policy
- Computes mean metrics per regime/policy and per-example improvement percentages
- Runs paired t-tests and effect-size calculations across baseline comparisons
- Flags data-quality issues such as identical baseline outputs and hardcoded coverage

The notebook is designed to run locally and in Google Colab. Data is loaded from a GitHub URL with a local fallback.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('loguru==0.7.3')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

## Setup

The cell above installs dependencies. It installs `loguru` everywhere, and installs core scientific packages only when NOT running on Colab, to avoid corrupting Colab's pre-installed compiled extensions.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from loguru import logger

# NumPy 2.0 compatibility shims
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

# Notebook-friendly logger: stdout only
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

## Data Loading

Load experiment results from GitHub first, then fall back to a local `mini_demo_data.json` if the network is unavailable.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-087852-visual-uncertainty-aware-conformal-cache/main/round-2/evaluation-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
logger.info(f"Loaded data with {len(data.get('datasets', []))} dataset(s)")
if data.get('datasets'):
    examples = data['datasets'][0].get('examples', [])
    logger.info(f"Loaded {len(examples)} examples")

## Config

Tunable parameters. Start at minimum values.

In [ ]:
# Absolute minimum demo parameters
MAX_EXAMPLES = None          # None => use all loaded examples
PLOT_TOP_N_METRICS = 8       # how many metrics to show in summary plots
SAVE_OUTPUT = False          # set True to write full_eval_out.json

## Processing: Load and organize experiment results

In [ ]:
# Extract datasets from loaded data
datasets = data.get("datasets", [])
if not datasets:
    logger.error("No datasets found in experiment results")
    sys.exit(1)

dataset = datasets[0]
examples = dataset.get("examples", [])
if MAX_EXAMPLES is not None:
    examples = examples[:MAX_EXAMPLES]
logger.info(f"Using {len(examples)} examples for evaluation")

## Processing: Parse outputs and build evaluation records

In [ ]:
# Organize data by regime, policy, chunk
# Structure: data[regime][policy][chunk] = parsed_output_dict
data_by_regime_policy_chunk = {}
raw_examples = []

for ex in examples:
    input_str = ex["input"]
    output_str = ex["output"]
    try:
        output = json.loads(output_str)
    except json.JSONDecodeError:
        logger.error(f"Failed to parse output JSON: {output_str}")
        continue
    
    regime = ex.get("metadata_regime")
    policy = ex.get("metadata_policy")
    chunk = ex.get("metadata_chunk")
    
    if regime not in data_by_regime_policy_chunk:
        data_by_regime_policy_chunk[regime] = {}
    if policy not in data_by_regime_policy_chunk[regime]:
        data_by_regime_policy_chunk[regime][policy] = {}
    data_by_regime_policy_chunk[regime][policy][chunk] = output
    
    # Keep raw example for output, but add per-example evaluation metrics
    eval_ex = ex.copy()
    # Parse predict_* fields to compute improvement metrics
    try:
        pred_lru = float(ex.get("predict_LRU", 0))
        pred_tinylfu = float(ex.get("predict_TinyLFU", 0))
        pred_arc2 = float(ex.get("predict_ARC2", 0))
        pred_ours = float(ex.get("predict_OURS", 0))
        
        # Avoid division by zero
        if pred_lru > 0:
            eval_ex["eval_improvement_over_lru"] = (pred_ours - pred_lru) / pred_lru * 100.0
        else:
            eval_ex["eval_improvement_over_lru"] = 0.0
            
        if pred_tinylfu > 0:
            eval_ex["eval_improvement_over_tinylfu"] = (pred_ours - pred_tinylfu) / pred_tinylfu * 100.0
        else:
            eval_ex["eval_improvement_over_tinylfu"] = 0.0
            
        if pred_arc2 > 0:
            eval_ex["eval_improvement_over_arc2"] = (pred_ours - pred_arc2) / pred_arc2 * 100.0
        else:
            eval_ex["eval_improvement_over_arc2"] = 0.0
    except (ValueError, TypeError):
        # If parsing fails, set default values
        eval_ex["eval_improvement_over_lru"] = 0.0
        eval_ex["eval_improvement_over_tinylfu"] = 0.0
        eval_ex["eval_improvement_over_arc2"] = 0.0
    
    raw_examples.append(eval_ex)

logger.info(f"Regimes found: {list(data_by_regime_policy_chunk.keys())}")
for regime in data_by_regime_policy_chunk:
    logger.info(f"  Policies: {list(data_by_regime_policy_chunk[regime].keys())}")
    for policy in data_by_regime_policy_chunk[regime]:
        logger.info(f"    Chunks: {sorted(data_by_regime_policy_chunk[regime][policy].keys())}")

## Processing: Compute aggregate metrics

In [ ]:
# Metrics to compute
metrics = ["hit_ratio", "throughput", "average_latency", "coverage_error"]
regimes = list(data_by_regime_policy_chunk.keys())
policies = ["LRU", "TinyLFU", "ARC2", "OURS"]
baselines = ["LRU", "TinyLFU", "ARC2"]

# Prepare metrics_agg dictionary
metrics_agg = {}

# Compute mean metrics per regime, policy
for regime in regimes:
    for policy in policies:
        if policy not in data_by_regime_policy_chunk[regime]:
            continue
        policy_data = data_by_regime_policy_chunk[regime][policy]
        chunks = sorted(policy_data.keys())
        for metric in metrics:
            values = [policy_data[chunk].get(metric, np.nan) for chunk in chunks]
            values = [v for v in values if not np.isnan(v)]
            mean_val = float(np.mean(values)) if values else float(np.nan)
            regime_key = regime.replace('-', '_')
            metric_name = f"{metric}_{regime_key}_{policy}"
            metrics_agg[metric_name] = mean_val

## Processing: Paired statistical tests vs baselines

In [ ]:
# Compute effect sizes, p-values, improvement percentages for each metric vs each baseline
for regime in regimes:
    for metric in metrics:
        for baseline in baselines:
            if baseline not in data_by_regime_policy_chunk[regime] or "OURS" not in data_by_regime_policy_chunk[regime]:
                continue
            baseline_chunks = sorted(data_by_regime_policy_chunk[regime][baseline].keys())
            ours_chunks = sorted(data_by_regime_policy_chunk[regime]["OURS"].keys())
            common_chunks = sorted(set(baseline_chunks) & set(ours_chunks))
            if len(common_chunks) < 2:
                logger.warning(f"Not enough common chunks for {regime} {metric} {baseline}")
                continue
            baseline_vals = [data_by_regime_policy_chunk[regime][baseline][c].get(metric, np.nan) for c in common_chunks]
            ours_vals = [data_by_regime_policy_chunk[regime]["OURS"][c].get(metric, np.nan) for c in common_chunks]
            pairs = [(b, o) for b, o in zip(baseline_vals, ours_vals) if not (np.isnan(b) or np.isnan(o))]
            if len(pairs) < 2:
                logger.warning(f"Not enough valid pairs for {regime} {metric} {baseline}")
                continue
            baseline_arr, ours_arr = zip(*pairs)
            baseline_arr = np.array(baseline_arr)
            ours_arr = np.array(ours_arr)
            
            differences = ours_arr - baseline_arr
            mean_diff = np.mean(differences)
            std_diff = np.std(differences, ddof=1)
            effect_size = mean_diff / std_diff if std_diff != 0 else np.nan
            
            try:
                t_stat, p_val = stats.ttest_rel(ours_arr, baseline_arr)
            except Exception as e:
                logger.warning(f"Paired t-test failed: {e}")
                p_val = np.nan
            
            baseline_mean = np.mean(baseline_arr)
            improvement_pct = (mean_diff / baseline_mean) * 100.0 if baseline_mean != 0 else np.nan
            
            regime_key = regime.replace('-', '_')
            metrics_agg[f"effect_size_{metric}_{regime_key}_vs_{baseline}"] = float(effect_size)
            metrics_agg[f"p_value_{metric}_{regime_key}_vs_{baseline}"] = float(p_val)
            metrics_agg[f"improvement_percentage_{metric}_{regime_key}_vs_{baseline}"] = float(improvement_pct)

## Processing: Data-quality checks

In [ ]:
# 1. identical-baseline-results
identical_baseline = True
for regime in regimes:
    for policy in baselines:
        if policy not in data_by_regime_policy_chunk[regime]:
            identical_baseline = False
            break
        if policy == baselines[0]:
            continue
        ref_policy = baselines[0]
        ref_chunks = sorted(data_by_regime_policy_chunk[regime][ref_policy].keys())
        curr_chunks = sorted(data_by_regime_policy_chunk[regime][policy].keys())
        if ref_chunks != curr_chunks:
            identical_baseline = False
            break
        for chunk in ref_chunks:
            ref_vals = data_by_regime_policy_chunk[regime][ref_policy][chunk]
            curr_vals = data_by_regime_policy_chunk[regime][policy][chunk]
            for metric in metrics:
                if ref_vals.get(metric) != curr_vals.get(metric):
                    identical_baseline = False
                    break
            if not identical_baseline:
                break
        if not identical_baseline:
            break
    if not identical_baseline:
        break
metrics_agg["data_quality_identical_baseline_results"] = float(identical_baseline)

# 2. hardcoded-coverage
hardcoded_coverage = True
for regime in regimes:
    for policy in policies:
        if policy not in data_by_regime_policy_chunk[regime]:
            continue
        policy_data = data_by_regime_policy_chunk[regime][policy]
        chunks = sorted(policy_data.keys())
        cov_errors = [policy_data[chunk].get("coverage_error", np.nan) for chunk in chunks]
        cov_errors = [v for v in cov_errors if not np.isnan(v)]
        if not cov_errors:
            continue
        if len(set(cov_errors)) > 1:
            hardcoded_coverage = False
            break
    if not hardcoded_coverage:
        break
metrics_agg["data_quality_hardcoded_coverage"] = float(hardcoded_coverage)

# 3. vucca_underperforming_baselines
vucca_underperforming = True
for regime in regimes:
    if "OURS" not in data_by_regime_policy_chunk[regime]:
        vucca_underperforming = False
        break
    ours_data = data_by_regime_policy_chunk[regime]["OURS"]
    for baseline in baselines:
        if baseline not in data_by_regime_policy_chunk[regime]:
            vucca_underperforming = False
            break
        baseline_data = data_by_regime_policy_chunk[regime][baseline]
        common_chunks = sorted(set(ours_data.keys()) & set(baseline_data.keys()))
        for chunk in common_chunks:
            ours_hr = ours_data[chunk].get("hit_ratio", np.nan)
            base_hr = baseline_data[chunk].get("hit_ratio", np.nan)
            if np.isnan(ours_hr) or np.isnan(base_hr):
                continue
            if ours_hr >= base_hr:
                vucca_underperforming = False
                break
        if not vucca_underperforming:
            break
    if not vucca_underperforming:
        break
metrics_agg["data_quality_vucca_underperforming_baselines"] = float(vucca_underperforming)

## Results: Summary and Visualization

In [ ]:
output = {
    "metadata": {
        "evaluation_name": "Statistical Evaluation of VUCCA Cache Results",
        "description": "Comprehensive statistical evaluation of Visual Uncertainty-Aware Conformal Cache Admission experiment results",
        "regimes": regimes,
        "policies": policies,
        "metrics_computed": metrics,
    },
    "metrics_agg": metrics_agg,
    "datasets": [
        {
            "dataset": dataset.get("dataset", "cache_admission_evaluation"),
            "examples": raw_examples
        }
    ]
}

if SAVE_OUTPUT:
    output_path = Path("full_eval_out.json")
    logger.info(f"Saving evaluation results to {output_path}")
    with output_path.open("w") as f:
        json.dump(output, f, indent=2)

# Print a compact summary of aggregate metrics
print("\n=== Aggregate Metrics Summary ===")
for k, v in sorted(metrics_agg.items()):
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

In [ ]:
# Plot selected aggregate metrics for quick inspection
keys_to_plot = [k for k in metrics_agg.keys() if not k.startswith("data_quality_")][:PLOT_TOP_N_METRICS]
values_to_plot = [metrics_agg[k] for k in keys_to_plot]

plt.figure(figsize=(10, 6))
plt.barh(range(len(keys_to_plot)), values_to_plot, color="steelblue")
plt.yticks(range(len(keys_to_plot)), keys_to_plot, fontsize=8)
plt.xlabel("Value")
plt.title("Selected Aggregate Metrics")
plt.tight_layout()
plt.show()